In [1]:
#!pip install aiosqlite matplotlib seaborn --q


In [2]:
import asyncio       # Enables asynchronous programming using async/await; 
                     # used here to process LLM extraction requests concurrently in batches.
import json          # Handles parsing and serializing JSON data; 
                     # used in our custom logger to build a clean JSON array log file.
import logging       # Python's built-in logging framework for tracking application events, 
                     # warnings, and errors.

import sqlite3       # Python's built-in relational database library for synchronous operations 
                     # (used for database initialization, querying totals, and chart rendering).
from datetime import date  # Provides a native Python calendar date object (YYYY-MM-DD), 
                           # used to enforce strict typing for transaction dates.
import aiosqlite     # An asynchronous wrapper for SQLite3; 
                     # allows non-blocking database writes concurrently alongside async LLM calls.
from openai import AsyncOpenAI  # The official asynchronous client for OpenAI's API, 
                                # allowing non-blocking requests to models like gpt-4o-mini.
from pydantic import BaseModel, Field, field_validator  # Data validation library using Python type hints; 
                                                        # guarantees structured schema enforcement and parses dates safely.
from pythonjsonlogger import jsonlogger                  # Formats standard log output into structured JSON objects 
                                                         # for better log management and parsing.
import matplotlib.pyplot as plt  # Plotting library used to query the database 
                                 # and generate/save a visual bar chart of expenses by category.
import pandas as pd              # Data manipulation library

In [3]:
# Custom logging handler to collect log entries in memory and save them 
# as a single valid JSON array ([...]) rather than standard newline-delimited lines.
class JSONArrayListHandler(logging.Handler):
    
    def __init__(self, filename="expenses.json"):
        """Initializes the custom handler with a target filename, 
        an empty list to store log records, and a JSON formatter."""
        super().__init__()
        self.filename = filename
        self.records = []  # In-memory storage for all captured log dictionaries
        
        # Sets up python-json-logger with standard timestamp, level, logger name, and message keys
        self.formatter = jsonlogger.JsonFormatter(
            "%(asctime)s %(levelname)s %(name)s %(message)s"
        )

    def emit(self, record):
        """Automatically called whenever a log message is generated. 
        Intercepts the log record, formats it into JSON, and appends it to memory."""
        try:
            # Format the log record into a JSON string using the JSON formatter
            formatted_msg = self.formatter.format(record)
            
            # Parse the JSON string back into a Python dictionary and store it in our list
            self.records.append(json.loads(formatted_msg))
        except Exception:
            # Standard Python logging error handler fallback if formatting fails
            self.handleError(record)

    def flush_to_file(self):
        """Dumps all collected log dictionaries from memory into the target file 
        as a properly structured and indented JSON array."""
        with open(self.filename, "w", encoding="utf-8") as f:
            # Serializes the Python list of dicts into a formatted JSON array with an indent of 4
            json.dump(self.records, f, indent=4)

In [4]:
# 1. Retrieve or create a named logger instance ('expense_tracker') 
# to isolate our application's logs from other third-party libraries.
logger = logging.getLogger("expense_tracker")

# 2. Set the logging threshold to INFO. 
# This ensures that INFO, WARNING, ERROR, and CRITICAL logs are captured, 
# while noisier DEBUG messages are ignored.
logger.setLevel(logging.INFO)

# 3. Instantiate our custom JSON array logging handler, 
# directing it to prepare 'expenses.json' as the target output file.
json_array_handler = JSONArrayListHandler("expenses.json")

# 4. Attach the custom handler to our logger. 
# From this point on, any log message generated by 'logger' will be 
# intercepted by this handler, formatted, and stored in memory.
logger.addHandler(json_array_handler)

In [5]:
date

datetime.date

In [6]:
client = AsyncOpenAI()

In [7]:
# Defines a structured Pydantic data model to validate, type-cast, 
# and ensure clean extraction of expense details from LLM outputs.
class ExpenseSchema(BaseModel):
    
    # Mandatory field enforcing a Python 'date' object (YYYY-MM-DD format).
    # The description acts as prompt engineering instructions for the LLM.
    transaction_date: date = Field(
        description="Date of the transaction mentioned in the input or inferred (YYYY-MM-DD format). If unknown, use today's date."
    )
    
    # Mandatory field capturing the numerical value of the transaction amount.
    amount: float = Field(
        description="Numerical value of the expense amount"
    )
    
    # Mandatory field capturing the currency identifier (e.g., INR, USD, Rs, EUR).
    currency: str = Field(
        description="Currency code or symbol, e.g., INR, USD, Rs, EUR"
    )
    
    # Mandatory category classification field for grouping and visualization.
    transaction_type: str = Field(
        description="Category such as Travel Expense, Food Expense, Monthly Rent, Online Shopping, Utilities, Groceries, etc."
    )
    
    # Mandatory short summary text describing the nature of the expense.
    narrative: str = Field(
        description="A concise narrative or summary description of the expense"
    )

    # Custom Pydantic validator that runs BEFORE standard type checking ('mode='before'').
    @field_validator('transaction_date', mode='before')
    @classmethod
    def parse_date(cls, value):
        """Safely handles missing, unknown, or blank dates returned by the LLM 
        by falling back automatically to today's current date."""
        
        # Check if the incoming value is a string and matches 'unknown' or is empty/whitespace
        if isinstance(value, str) and (value.lower() == "unknown" or not value.strip()):
            return date.today()  # Fall back to today's date
            
        # Otherwise, pass the original value through for standard parsing into a date object
        return value

In [8]:
MOCK_USER_INPUTS = [
    "I travelled from home to office in Uber and paid 100 Rs for the ride today.",
    "Ordered a large pizza from Domino's last night for 450 INR.",
    "Paid my monthly apartment rent of 25000 Rs via bank transfer this morning.",
    "Bought a new wireless mouse on Amazon for 1299 INR during the flash sale.",
    "Filled petrol in my car at the local station costing 2000 Rs.",
    "Had a team lunch at Starbucks yesterday, bill came out to 850 INR.",
    "Paid the monthly electricity bill of 3200 Rs online.",
    "Purchased a Python programming book from Flipkart for 650 INR.",
    "Booked a metro ticket to the airport for 60 Rs.",
    "Bought groceries and vegetables from the supermarket worth 1850 INR.",
    "Went to the local cinema with friends, movie tickets cost 500 Rs each.",
    "Subscribed to Netflix monthly plan for 649 INR.",
    "Bought a cup of cappuccino at Café Coffee Day for 220 Rs.",
    "Paid broadband internet bill of 999 Rs for this month.",
    "Purchased a new cotton shirt from Myntra for 1499 INR.",
    "Paid cab fare using Ola to reach the railway station amounting to 350 Rs.",
    "Ordered Chinese takeout from Zomato for dinner costing 720 INR.",
    "Bought a pack of medicines from the local pharmacy for 310 Rs.",
    "Paid gym membership fee for the month amounting to 1500 Rs.",
    "Bought concert tickets online for 2500 INR."
]

In [9]:
DB_NAME = "expenses.db"

def init_db():
    """Initializes the SQLite database and creates the expenses table synchronously."""
    
    # Opens a synchronous connection to the SQLite database file ('expenses.db').
    # Using a context manager ('with') ensures the connection automatically closes 
    # when exiting the block, even if an exception occurs.
    with sqlite3.connect(DB_NAME) as conn:
        
        # Executes an SQL command to create the 'expenses' table if it does not already exist.
        conn.execute("""
            CREATE TABLE IF NOT EXISTS expenses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,  -- Auto-incrementing unique primary key for each row
                transaction_date TEXT,                 -- Date stored as a readable ISO string (YYYY-MM-DD)
                amount REAL,                           -- Numerical expense amount (stored as float/real)
                currency TEXT,                         -- Currency code or symbol (e.g., INR, USD, Rs)
                transaction_type TEXT,                 -- Category classification (e.g., Travel Expense, Food Expense)
                narrative TEXT,                        -- Short descriptive summary of the expense
                raw_input TEXT                         -- Original natural language text provided by the user
            )
        """)
        
        # Explicitly commits the table creation transaction to the database.
        conn.commit()
        
    # Logs a successful initialization event to our custom logger.
    logger.info("Database initialized and table created.")

In [10]:
# Asynchronous function to process a single natural language expense input, 
# extract details via OpenAI, validate with Pydantic, and save to SQLite.
async def process_and_store_expense(user_input: str):
    """Extracts using standard Chat Completions, validates with Pydantic, and saves asynchronously."""
    try:
        # 1. Log that processing has started for this specific input string
        logger.info("Processing user input via LLM", extra={"input_text": user_input})
        
        # 2. Define strict instructions for the LLM to guarantee a structured JSON output
        SYSTEM_PROMPT = """
        You are an expert financial assistant. Extract transaction details accurately from the user's natural language input and return a JSON object 
        with these exact keys: 
        'transaction_date' (string in YYYY-MM-DD format, or 'Unknown'), 
        'amount' (float), 
        'currency' (string, e.g. INR, USD, Rs), 
        'transaction_type' (string, e.g. Travel Expense, Food Expense, Monthly Rent, Online Shopping), 
        and 'narrative' (string summary description).

        Make sure that the currency is standardized to ISO currency and if there a reference made to 
        date as last day or last month or any other reference, ensure that you make reference with
        respect to the current year 2026 only and current date.
        """
        
        # 3. Call the OpenAI Chat Completions API asynchronously using gpt-4o-mini.
        # 'response_format={"type": "json_object"}' forces the model to return valid JSON.
        completion = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT,
                },
                {"role": "user", "content": user_input},
            ],
            response_format={"type": "json_object"},
        )
        
        # 4. Extract the raw JSON text content from the API response message
        response_content = completion.choices[0].message.content
        
        # 5. Parse and validate the JSON string against our Pydantic schema.
        # This automatically triggers our custom field validator if the date is 'Unknown' or missing.
        expense: ExpenseSchema = ExpenseSchema.model_validate_json(response_content)
        
        # 6. Convert the validated Pydantic date object into a standard ISO string (YYYY-MM-DD)
        date_str = expense.transaction_date.isoformat()
        
        # 7. Open an asynchronous connection to SQLite using aiosqlite 
        # so database writes don't block other concurrent network calls.
        async with aiosqlite.connect(DB_NAME) as db:
            await db.execute(
                """
                INSERT INTO expenses (transaction_date, amount, currency, transaction_type, narrative, raw_input)
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    date_str,
                    expense.amount,
                    expense.currency,
                    expense.transaction_type,
                    expense.narrative,
                    user_input,  # Store the original text for reference/auditing
                ),
            )
            # Commit the transaction asynchronously to save changes to the database file
            await db.commit()
            
        # 8. Log a success message with structured metadata about the processed expense
        logger.info(
            "Successfully validated and saved expense asynchronously",
            extra={
                "category": expense.transaction_type,
                "expense_amount": expense.amount,
                "curr": expense.currency,
                "tx_date": date_str
            }
        )
        
    except Exception as e:
        # 9. Catch any errors (e.g., API rate limits, network timeouts, JSON formatting errors) 
        # and log them with the associated user input for debugging.
        logger.error("Error processing input", extra={"input_text": user_input, "error": str(e)})

In [11]:
# Asynchronous pipeline function to process a large list of inputs in controlled chunks (batches) 
# to prevent overwhelming rate limits and database connections.
async def run_async_batch_pipeline(inputs: list[str], batch_size: int = 5):
    """Processes inputs in controlled batches using asyncio.gather for each batch."""
    
    # 1. Determine the total number of inputs to process
    total_inputs = len(inputs)
    
    # 2. Loop through the input list in steps defined by 'batch_size' (default is 5)
    for i in range(0, total_inputs, batch_size):
        
        # Slicing the list to get the current chunk of inputs (e.g., items 0 to 5, 5 to 10, etc.)
        batch = inputs[i:i + batch_size]
        
        # Calculate the current batch number (starting from 1) for logging and progress tracking
        batch_num = (i // batch_size) + 1
        
        # Log the start of the batch with metadata (using custom keys to avoid reserved LogRecord conflicts)
        logger.info(f"Processing batch {batch_num}", extra={"chunk_size": len(batch), "start_idx": i})
        
        # 3. Create a list of asynchronous tasks for each text item in the current batch
        tasks = [process_and_store_expense(text) for text in batch]
        
        # 4. Execute all tasks in the current batch concurrently and wait until all of them complete
        await asyncio.gather(*tasks)
        
        # 5. Print a clear progress update to the console showing how many total items have been processed so far
        print(f"Completed batch {batch_num} ({min(i + batch_size, total_inputs)}/{total_inputs} processed).")

In [12]:
# Synchronous function to query the database, calculate category totals, 
# and render a visual bar chart of expenses.
def generate_visualization():
    """Queries the SQLite database and generates a bar chart of expenses by transaction type."""
    
    # 1. Connect to the SQLite database synchronously using a context manager
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        
        # 2. Execute an SQL query to group expenses by category ('transaction_type') 
        # and calculate the sum of amounts for each group.
        cursor.execute(
            "SELECT transaction_type, SUM(amount) FROM expenses GROUP BY transaction_type"
        )
        
        # Fetch all aggregated rows from the query result
        data = cursor.fetchall()

    # 3. Check if the database returned any records; if empty, exit early
    if not data:
        print("No data available to generate visualization.")
        return

    # 4. Separate the query results into two distinct lists for plotting: 
    # category labels on the X-axis and numerical sums on the Y-axis.
    types = [row[0] for row in data]
    amounts = [row[1] for row in data]

    # 5. Initialize a new Matplotlib figure canvas with custom dimensions (10x6 inches)
    plt.figure(figsize=(10, 6))
    
    # Render a bar chart with sky-blue bars and black borders for visual clarity
    plt.bar(types, amounts, color='skyblue', edgecolor='black')
    
    # Label the X-axis, Y-axis, and add a descriptive chart title
    plt.xlabel('Transaction Type', fontsize=12)
    plt.ylabel('Total Amount', fontsize=12)
    plt.title('Total Expenses by Transaction Type', fontsize=14)
    
    # Rotate X-axis category labels by 45 degrees and align them to the right 
    # to prevent text overlapping if category names are long.
    plt.xticks(rotation=45, ha='right')
    
    # Adjust layout formatting to ensure labels and titles fit neatly without getting clipped
    plt.tight_layout()
    
    # 6. Save the generated chart image to disk
    chart_filename = "expenses_by_type.png"
    plt.savefig(chart_filename)
    
    # Close the Matplotlib plot to free up system memory resources
    plt.close()
    
    # 7. Log the successful chart creation using 'chart_file' (avoiding reserved LogRecord keys) 
    # and print a confirmation message to the console.
    logger.info("Visualization generated successfully", extra={"chart_file": chart_filename})
    print(f"Chart successfully generated and saved as '{chart_filename}'.")

In [13]:
# Top-level asynchronous orchestration function that coordinates the entire application workflow.
async def process_trans():
    
    # 1. Initialize the SQLite database and create the 'expenses' table synchronously 
    # before any data extraction or insertion begins.
    init_db()
    
    # 2. Notify the user that batch processing is starting, then invoke 
    # the asynchronous pipeline to process mock inputs in chunks of 5 concurrently.
    print("Starting asynchronous batch extraction, validation, and database saving...")
    await run_async_batch_pipeline(MOCK_USER_INPUTS, batch_size=5)
    print("All 20 transactions processed and saved in batches.")
    
    # 3. Query the populated SQLite database to aggregate expenses by category 
    # and render/save the visual Matplotlib bar chart.
    print("Generating expense chart visualization...")
    generate_visualization()
    
    # 4. Flush all log entries collected in memory by our custom handler 
    # and write them out to 'expenses.json' as a clean, valid JSON array.
    json_array_handler.flush_to_file()
    print("Logs successfully flushed to 'expenses.json' as a JSON array.")

In [14]:
await process_trans()

Starting asynchronous batch extraction, validation, and database saving...
Completed batch 1 (5/20 processed).
Completed batch 2 (10/20 processed).
Completed batch 3 (15/20 processed).
Completed batch 4 (20/20 processed).
All 20 transactions processed and saved in batches.
Generating expense chart visualization...
Chart successfully generated and saved as 'expenses_by_type.png'.
Logs successfully flushed to 'expenses.json' as a JSON array.


In [15]:
def query_expenses_db(sql_query: str, params: tuple = ()) -> list:
    """
    Connects to the expenses database, executes a custom SQL query, 
    and returns all fetched records.
    
    Args:
        sql_query (str): The SQL query string to execute.
        params (tuple): Optional parameters for parameterized queries (helps prevent SQL injection).
        
    Returns:
        list: A list of tuples containing the fetched rows.
    """
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        # Execute the query with optional parameters
        cursor.execute(sql_query, params)
        # Fetch and return all rows resulting from the query
        return cursor.fetchall()

In [16]:
query_expenses_db("select * from expenses")

[(1,
  '2026-10-06',
  450.0,
  'INR',
  'Food Expense',
  "Ordered a large pizza from Domino's.",
  "Ordered a large pizza from Domino's last night for 450 INR."),
 (2,
  '2026-10-06',
  100.0,
  'INR',
  'Travel Expense',
  'Uber ride from home to office.',
  'I travelled from home to office in Uber and paid 100 Rs for the ride today.'),
 (3,
  '2026-09-30',
  25000.0,
  'INR',
  'Monthly Rent',
  'Paid monthly apartment rent via bank transfer.',
  'Paid my monthly apartment rent of 25000 Rs via bank transfer this morning.'),
 (4,
  '2026-10-19',
  2000.0,
  'INR',
  'Fuel Expense',
  'Filled petrol in car at local station.',
  'Filled petrol in my car at the local station costing 2000 Rs.'),
 (5,
  '2026-10-05',
  1299.0,
  'INR',
  'Online Shopping',
  'Purchased a wireless mouse on Amazon during a flash sale.',
  'Bought a new wireless mouse on Amazon for 1299 INR during the flash sale.'),
 (6,
  '2026-10-04',
  850.0,
  'INR',
  'Food Expense',
  'Team lunch at Starbucks.',
  'Ha

In [17]:
def display_expenses_table(sql_query: str = "SELECT * FROM expenses", params: tuple = ()):
    """
    Queries the SQLite database and returns a pandas DataFrame, 
    which renders as a formatted table in Jupyter Notebooks.
    
    Args:
        sql_query (str): The SQL query string to execute.
        params (tuple): Optional parameters for parameterized queries.
        
    Returns:
        pd.DataFrame: The query results formatted as a DataFrame table.
    """
    with sqlite3.connect(DB_NAME) as conn:
        # pd.read_sql_query automatically handles column names and table formatting
        df = pd.read_sql_query(sql_query, conn, params=params)
        
    return df

In [18]:
display_expenses_table()

,id,transaction_date,amount,currency,transaction_type,narrative,raw_input
0,1,2026-10-06,450.0,INR,Food Expense,Ordered a large pizza from Domino's.,Ordered a large pizza from Domino's last night...
1,2,2026-10-06,100.0,INR,Travel Expense,Uber ride from home to office.,I travelled from home to office in Uber and pa...
2,3,2026-09-30,25000.0,INR,Monthly Rent,Paid monthly apartment rent via bank transfer.,Paid my monthly apartment rent of 25000 Rs via...
3,4,2026-10-19,2000.0,INR,Fuel Expense,Filled petrol in car at local station.,Filled petrol in my car at the local station c...
4,5,2026-10-05,1299.0,INR,Online Shopping,Purchased a wireless mouse on Amazon during a ...,Bought a new wireless mouse on Amazon for 1299...
5,6,2026-10-04,850.0,INR,Food Expense,Team lunch at Starbucks.,"Had a team lunch at Starbucks yesterday, bill ..."
6,7,2026-10-20,650.0,INR,Online Shopping,Purchased a Python programming book from Flipk...,Purchased a Python programming book from Flipk...
7,8,2026-10-04,1850.0,INR,Food Expense,Purchased groceries and vegetables from the su...,Bought groceries and vegetables from the super...
8,9,2026-10-29,3200.0,INR,Monthly Utility Expense,Paid the monthly electricity bill online.,Paid the monthly electricity bill of 3200 Rs o...
9,10,2026-10-14,60.0,INR,Travel Expense,Booked a metro ticket to the airport.,Booked a metro ticket to the airport for 60 Rs.
